In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_FactClaimLineItem")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_claim_line = "../../data_lake/silver/silver_claim_line_item/"
silver_eob_line = "../../data_lake/silver/silver_eob_line_item/"
silver_claim_diag = "../../data_lake/silver/silver_claim_diagnosis/"
silver_claim_proc = "../../data_lake/silver/silver_claim_procedure/"
gold_base_path = "../../data_lake/gold/fact_claim_line_item/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"


In [ ]:
df_claim_line = spark.read.format("parquet").load(silver_claim_line)
df_eob_line = spark.read.format("parquet").load(silver_eob_line)
df_claim_diag = spark.read.format("parquet").load(silver_claim_diag)
df_claim_proc = spark.read.format("parquet").load(silver_claim_proc)
df_dimpatient = spark.read.format("parquet").load(gold_dimpatient)


In [ ]:
df_line_item = (df_claim_line.alias("clm_line")
    .join(
        df_eob_line.alias("eob_line"),
        (col("clm_line.claim_id") == col("eob_line.claim_id")) &
        (col("clm_line.item_seq") == col("eob_line.item_seq").cast("integer")),
        "left"
    )
    .join(
        df_claim_diag.alias("diag"),
        (col("clm_line.claim_id") == col("diag.claim_id")) &
        (col("clm_line.diagnosis_seq_ref") == col("diag.diagnosis_seq").cast("integer")),
        "left"
    )
    .join(
        df_claim_proc.alias("proc"),
        (col("clm_line.claim_id") == col("proc.claim_id")) &
        (col("clm_line.procedure_seq_ref") == col("proc.procedure_seq").cast("integer")),
        "left"
    )
)


In [ ]:
cond_id_col = F.regexp_replace(col("diag.diagnosis_reference"), "^(Condition/|urn:uuid:)", "")
proc_id_col = F.regexp_replace(col("proc.procedure_reference"), "^(Procedure/|urn:uuid:)", "")

df_inter = (df_line_item
    .join(df_dimpatient.alias("pat"), col("eob_line.patient_id") == col("pat.patient_id"), "left")
    .select(
        F.conv(F.substring(F.md5(F.concat(col("clm_line.claim_id"), F.lit("_"), col("clm_line.item_seq").cast("string"))), 1, 15), 16, 10).cast("bigint").alias("claim_line_key"),
        F.conv(F.substring(F.md5(col("clm_line.claim_id")), 1, 15), 16, 10).cast("bigint").alias("claim_key"),
        col("clm_line.claim_id"),
        col("clm_line.item_seq"),
        F.conv(F.substring(F.md5(col("clm_line.encounter_id")), 1, 15), 16, 10).cast("bigint").alias("encounter_key"),
        col("clm_line.encounter_id"),
        col("pat.patient_key"),
        col("clm_line.item_type"),
        col("clm_line.service_code"),
        col("clm_line.service_code_system"),
        col("clm_line.service_description"),
        col("eob_line.net_amt").cast("double").alias("net_amt"),
        col("eob_line.submitted_amt").cast("double").alias("submitted_amt"),
        col("eob_line.allowed_amt").cast("double").alias("allowed_amt"),
        col("eob_line.provider_payment_amt").cast("double").alias("provider_payment_amt"),
        col("eob_line.coinsurance_amt").cast("double").alias("coinsurance_amt"),
        col("eob_line.deductible_amt").cast("double").alias("deductible_amt"),
        col("clm_line.is_billable"),
        F.when(cond_id_col.isNotNull(), F.conv(F.substring(F.md5(cond_id_col), 1, 15), 16, 10).cast("bigint")).alias("condition_key"),
        F.when(proc_id_col.isNotNull(), F.conv(F.substring(F.md5(proc_id_col), 1, 15), 16, 10).cast("bigint")).alias("procedure_key"),
        F.current_timestamp().alias("gold_timestamp")
    )
)


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
